# HyDE: Hypothetical Document Embeddings — Demo

This notebook demonstrates the core idea behind HyDE: when user queries
are phrased casually and source documents are written formally, direct
query-to-document retrieval underperforms. Generating a hypothetical
answer first — and searching with that instead — closes much of the gap.

The corpus and embedder here are intentionally lightweight (TF-IDF,
in-memory) so this notebook runs fully offline. The retrieval logic is
identical to what a production system would use with real embeddings
and a real LLM generator — see `hyde.py` for the swap-in points.


In [1]:
import sys
sys.path.insert(0, '.')

from hyde import (
    Embedder,
    InMemoryVectorStore,
    HydeRetriever,
    TemplateGenerator,
    compare_retrieval,
)
from demo_corpus import DOCUMENTS, TEST_QUERIES

print(f"Corpus size: {len(DOCUMENTS)} documents")
print(f"Test queries: {len(TEST_QUERIES)}")


Corpus size: 6 documents
Test queries: 6


## 1. Index the document corpus

Documents are written in formal documentation style. This is the corpus
a real support/knowledge-base system would search over.


In [2]:
embedder = Embedder()
vector_store = InMemoryVectorStore(embedder)
vector_store.index(DOCUMENTS)

for doc in DOCUMENTS:
    print(f"[{doc.doc_id}] {doc.text[:80]}...")


[doc_payment_retry] Recurring transaction failures may occur due to insufficient account balance, ex...
[doc_account_lockout] Repeated failed authentication attempts trigger a temporary account lockout as a...
[doc_notification_delay] Notification delivery latency can result from provider-side queuing, device conn...
[doc_data_export] Bulk data export requests are processed asynchronously and the resulting file is...
[doc_integration_sync] Third-party integration synchronization failures are most commonly caused by an ...
[doc_refund_window] Refund eligibility is determined by the policy in effect at the time of purchase...


## 2. Set up the HyDE retriever

Using `TemplateGenerator` here for offline reproducibility. In
production this is swapped for `AnthropicGenerator` (or any LLM client)
— see `hyde.py`. The retrieval logic downstream is unchanged
either way.


In [3]:
retriever = HydeRetriever(generator=TemplateGenerator())


## 3. Compare direct retrieval vs. HyDE retrieval

For each casual test query, we run both:
- **Direct**: embed the raw query, search directly
- **HyDE**: generate a hypothetical document-style answer, embed that,
  search with it instead

We check whether the correct document lands as the top-1 result in
each case.


In [4]:
results_summary = []

for query, expected_doc_id in TEST_QUERIES:
    comparison = compare_retrieval(query, vector_store, retriever, top_k=3)

    direct_top1 = comparison["direct"][0].doc_id if comparison["direct"] else None
    hyde_top1 = comparison["hyde"][0].doc_id if comparison["hyde"] else None

    direct_correct = direct_top1 == expected_doc_id
    hyde_correct = hyde_top1 == expected_doc_id

    results_summary.append({
        "query": query,
        "expected": expected_doc_id,
        "direct_top1": direct_top1,
        "direct_correct": direct_correct,
        "hyde_top1": hyde_top1,
        "hyde_correct": hyde_correct,
    })

    print(f"Query: {query!r}")
    print(f"  Expected doc:      {expected_doc_id}")
    print(f"  Direct top-1:      {direct_top1}  {'✓' if direct_correct else '✗'}")
    print(f"  HyDE top-1:        {hyde_top1}  {'✓' if hyde_correct else '✗'}")
    print()


Query: 'why did my payment fail twice this month'
  Expected doc:      doc_payment_retry
  Direct top-1:      doc_refund_window  ✗
  HyDE top-1:        doc_payment_retry  ✓

Query: 'i keep getting locked out of my account'
  Expected doc:      doc_account_lockout
  Direct top-1:      doc_data_export  ✗
  HyDE top-1:        doc_account_lockout  ✓

Query: 'why are my notifications showing up late'
  Expected doc:      doc_notification_delay
  Direct top-1:      doc_refund_window  ✗
  HyDE top-1:        doc_notification_delay  ✓

Query: 'how long does it take to download all my data'
  Expected doc:      doc_data_export
  Direct top-1:      doc_data_export  ✓
  HyDE top-1:        doc_data_export  ✓

Query: 'my third party app stopped syncing'
  Expected doc:      doc_integration_sync
  Direct top-1:      doc_integration_sync  ✓
  HyDE top-1:        doc_integration_sync  ✓

Query: 'can i get my money back for this order'
  Expected doc:      doc_refund_window
  Direct top-1:      doc_refun

## 4. Aggregate accuracy: direct vs. HyDE


In [5]:
direct_accuracy = sum(r["direct_correct"] for r in results_summary) / len(results_summary)
hyde_accuracy = sum(r["hyde_correct"] for r in results_summary) / len(results_summary)

print(f"Direct retrieval top-1 accuracy: {direct_accuracy:.0%}")
print(f"HyDE retrieval top-1 accuracy:   {hyde_accuracy:.0%}")


Direct retrieval top-1 accuracy: 50%
HyDE retrieval top-1 accuracy:   100%


## 5. Inspect one example closely

Let's look at exactly what HyDE generated and why it changed the
retrieval outcome for one query.


In [6]:
example_query = TEST_QUERIES[0][0]
hypothetical = retriever.generator.generate(example_query)

print(f"Original query:\n  {example_query}\n")
print(f"Generated hypothetical document:\n  {hypothetical}\n")

comparison = compare_retrieval(example_query, vector_store, retriever, top_k=3)

print("Direct retrieval results:")
for r in comparison["direct"]:
    print(f"  {r.doc_id}  (score={r.score:.3f})")

print("\nHyDE retrieval results:")
for r in comparison["hyde"]:
    print(f"  {r.doc_id}  (score={r.score:.3f})")


Original query:
  why did my payment fail twice this month

Generated hypothetical document:
  Payment transaction failures are typically resolved through an automatic retry mechanism within a defined retry window, commonly triggered by insufficient balance, expired card credentials, or bank-side fraud flagging.

Direct retrieval results:
  doc_refund_window  (score=0.181)
  doc_integration_sync  (score=0.000)
  doc_data_export  (score=0.000)

HyDE retrieval results:
  doc_payment_retry  (score=0.626)
  doc_integration_sync  (score=0.182)
  doc_notification_delay  (score=0.046)


## 6. Ensemble HyDE (multiple hypothetical documents, averaged)

This smooths out noise from any single generation by averaging the
embeddings of several hypothetical documents.


In [7]:
ensemble_results = retriever.retrieve_ensemble(
    example_query, vector_store, n_hypotheses=4, top_k=3
)

print("Ensemble HyDE retrieval results:")
for r in ensemble_results:
    print(f"  {r.doc_id}  (score={r.score:.3f})")


Ensemble HyDE retrieval results:
  doc_payment_retry  (score=0.626)
  doc_integration_sync  (score=0.182)
  doc_notification_delay  (score=0.046)


## Notes on this demo vs. production

- **Embedder**: this notebook uses TF-IDF for reproducibility without
  API keys. In production, swap in a real embedding model — the
  `Embedder` class is the only thing that needs to change.
- **Generator**: this notebook uses `TemplateGenerator`, a deterministic
  offline stand-in. In production, swap in `AnthropicGenerator` (or any
  LLM client) — this is where the real gain comes from, since a real
  LLM's hypothetical answer captures actual domain vocabulary far
  better than a template can.
- **Corpus size**: six documents is enough to demonstrate the mechanism
  clearly. The effect is more pronounced, not less, on larger and more
  varied real-world corpora — small demo corpora tend to *understate*
  HyDE's benefit relative to production-scale collections.
